## Random Forest

Данные полученных экспериментов находятся в папке `results`.

In [105]:
# %pip install catboost

import polars as pl
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score, precision_recall_curve, auc, balanced_accuracy_score, accuracy_score

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

In [106]:
df_2020 = pd.read_csv(
    r'..\..\preprocessing\data\processed\df_2020_preprocessed.csv')

df_2020.head()

,HeartDisease,BMI,SmokerStatus,AlcoholDrinkers,HadStroke,PhysicalHealthDays,MentalHealthDays,DifficultyWalking,Sex,AgeCategory,RaceEthnicityCategory,HadDiabetes,PhysicalActivities,GeneralHealth,SleepHours,HadAsthma,HadKidneyDisease,HadSkinCancer
0,0,16.60,1,0,0,3.0,30.0,0,Female,55-59,White,Yes,1,Very good,5.0,1,0,1
1,0,20.34,0,0,1,0.0,0.0,0,Female,80 or older,White,No,1,Very good,7.0,0,0,0
2,0,26.58,1,0,0,20.0,30.0,0,Male,65-69,White,Yes,1,Fair,8.0,1,0,0
3,0,24.21,0,0,0,0.0,0.0,0,Female,75-79,White,No,0,Good,6.0,0,0,1
4,0,23.71,0,0,0,28.0,0.0,1,Female,40-44,White,No,1,Very good,8.0,0,0,0


In [107]:
results = pd.DataFrame()

### Обучение на наборе 2020 года

In [108]:
df_2020['AgeCategory'] = df_2020['AgeCategory'].map({
    '18-24': 0,
    '25-29': 1,
    '30-34': 2,
    '35-39': 3,
    '40-44': 4,
    '45-49': 5,
    '50-54': 6,
    '55-59': 7,
    '60-64': 8,
    '65-69': 9,
    '70-74': 10,
    '75-79': 11,
    '80 or older': 12,
})

df_2020['GeneralHealth'] = df_2020['GeneralHealth'].map({
    'Poor': 0,
    'Fair': 1,
    'Good': 2,
    'Very good': 3,
    'Excellent': 4,
})

In [109]:
SEED = 67

X = df_2020.drop('HeartDisease', axis=1)
y = df_2020['HeartDisease']
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    train_size=0.8,
    random_state=SEED,
    stratify=y
)

X_valid, X_test, y_valid, y_test = train_test_split(
    X_test,
    y_test,
    train_size=0.5,
    random_state=SEED,
    stratify=y_test
)

In [110]:
transformer = ColumnTransformer(
    transformers=[
        ('categories', OneHotEncoder(handle_unknown='ignore'),
         ['Sex', 'RaceEthnicityCategory', 'HadDiabetes']),
        ('numericals', Pipeline([
            ('nans', SimpleImputer(strategy='median')),
        ]), list(set(df_2020.columns) - {'Sex', 'RaceEthnicityCategory', 'HadDiabetes', 'HeartDisease'}))
    ]
)

model2020 = Pipeline([
    ('transformer', transformer),
    ('model', RandomForestClassifier(
        class_weight='balanced',
        random_state=SEED,
        n_jobs=-1,
        max_depth=15,
        min_samples_split=20,
        min_samples_leaf=10,
    ))
])

In [111]:
model2020.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('transformer', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('categories', ...), ('numericals', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different tr

In [112]:
y_proba2020 = model2020.predict_proba(X_test)[:, 1]
y_pred2020 = model2020.predict(X_test)

In [113]:
precision, recall, thresholds = precision_recall_curve(y_test, y_proba2020)

metrics2020 = {
    'data': '2020',
    'model': 'RandomForestClassifier',
    'roc_auc_score': roc_auc_score(y_test, y_proba2020),
    'f1_score': f1_score(y_test, y_pred2020),
    'pr_auc_score': auc(recall, precision),
    'balanced_accuracy_score': balanced_accuracy_score(y_test, y_pred2020)
}

In [114]:
metrics2020

{'data': '2020',
 'model': 'RandomForestClassifier',
 'roc_auc_score': 0.8393185812942191,
 'f1_score': 0.35626964722319243,
 'pr_auc_score': 0.33593313917241774,
 'balanced_accuracy_score': 0.7626956307593167}

### Обучение на общем наборе 2020+2022

In [115]:
df_merged = pd.read_csv(r'..\..\preprocessing\data\processed\df_merged.csv')
df_merged.head()

,HeartDisease,BMI,SmokerStatus,AlcoholDrinkers,HadStroke,PhysicalHealthDays,MentalHealthDays,DifficultyWalking,Sex,AgeCategory,RaceEthnicityCategory,HadDiabetes,PhysicalActivities,GeneralHealth,SleepHours,HadAsthma,HadKidneyDisease,HadSkinCancer,Year
0,0,15.0,0.0,0.0,0.0,0.0,0.0,0.0,Female,50-54,Hispanic,No,1.0,Fair,7.0,0.0,0.0,0.0,2020
1,0,15.0,0.0,0.0,0.0,0.0,0.0,0.0,Female,80 or older,White,No,1.0,Excellent,8.0,0.0,0.0,1.0,2020
2,0,15.0,0.0,0.0,0.0,0.0,0.0,0.0,Male,70-74,White,Yes,1.0,Good,8.0,0.0,0.0,0.0,2020
3,0,15.0,0.0,1.0,0.0,0.0,0.0,0.0,Male,50-54,White,No,0.0,Excellent,9.0,0.0,0.0,0.0,2022
4,0,15.0,0.0,1.0,0.0,0.0,3.0,0.0,Female,55-59,White,No,1.0,Very good,6.0,0.0,0.0,0.0,2022


In [116]:
df_merged['AgeCategory'] = df_merged['AgeCategory'].map({
    '18-24': 0,
    '25-29': 1,
    '30-34': 2,
    '35-39': 3,
    '40-44': 4,
    '45-49': 5,
    '50-54': 6,
    '55-59': 7,
    '60-64': 8,
    '65-69': 9,
    '70-74': 10,
    '75-79': 11,
    '80 or older': 12,
})

df_merged['GeneralHealth'] = df_merged['GeneralHealth'].map({
    'Poor': 0,
    'Fair': 1,
    'Good': 2,
    'Very good': 3,
    'Excellent': 4,
})

In [117]:
SEED = 67

X = df_merged.drop('HeartDisease', axis=1)
y = df_merged['HeartDisease']
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    train_size=0.8,
    random_state=SEED,
    stratify=y
)

X_valid, X_test, y_valid, y_test = train_test_split(
    X_test,
    y_test,
    train_size=0.5,
    random_state=SEED,
    stratify=y_test
)

In [118]:
transformer_merged = ColumnTransformer(
    transformers=[
        ('categories', OneHotEncoder(handle_unknown='ignore'),
         ['Sex', 'RaceEthnicityCategory', 'HadDiabetes']),
        ('numericals', Pipeline([
            ('nans', SimpleImputer(strategy='median')),
        ]), list(set(df_merged.columns) - {'Sex', 'RaceEthnicityCategory', 'HadDiabetes', 'HeartDisease'}))
    ]
)

model_merged = Pipeline([
    ('transformer', transformer_merged),
    ('model', RandomForestClassifier(
        class_weight='balanced',
        random_state=SEED,
        n_jobs=-1,
        max_depth=15,
        min_samples_split=20,
        min_samples_leaf=10,
    ))
])

model_merged.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('transformer', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('categories', ...), ('numericals', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different tr

In [119]:
y_proba_merged = model_merged.predict_proba(X_test)[:, 1]
y_pred_merged = model_merged.predict(X_test)

precision, recall, thresholds = precision_recall_curve(y_test, y_proba_merged)

metrics_merged = {
    'data': 'merged',
    'model': 'RandomForestClassifier',
    'roc_auc_score': roc_auc_score(y_test, y_proba_merged),
    'f1_score': f1_score(y_test, y_pred_merged),
    'pr_auc_score': auc(recall, precision),
    'balanced_accuracy_score': balanced_accuracy_score(y_test, y_pred_merged)
}

In [120]:
metrics_merged

{'data': 'merged',
 'model': 'RandomForestClassifier',
 'roc_auc_score': 0.8350595400275327,
 'f1_score': 0.352980801616706,
 'pr_auc_score': 0.33867351768400367,
 'balanced_accuracy_score': 0.7574544954163691}

### Обучение на наборе 2022 года

In [121]:
df_2022 = pd.read_csv(
    r'..\..\preprocessing\data\processed\df_2022_preprocessed.csv')

df_2022['AgeCategory'] = df_2022['AgeCategory'].map({
    '18-24': 0,
    '25-29': 1,
    '30-34': 2,
    '35-39': 3,
    '40-44': 4,
    '45-49': 5,
    '50-54': 6,
    '55-59': 7,
    '60-64': 8,
    '65-69': 9,
    '70-74': 10,
    '75-79': 11,
    '80 or older': 12,
})

df_2022['GeneralHealth'] = df_2022['GeneralHealth'].map({
    'Poor': 0,
    'Fair': 1,
    'Good': 2,
    'Very good': 3,
    'Excellent': 4,
})

df_2022['RemovedTeeth'] = df_2022['RemovedTeeth'].map({
    'None of them': 0,
    '1 to 5': 1,
    '6 or more, but not all': 2,
    'All': 3
})

df_2022['LastCheckupTime'] = df_2022['LastCheckupTime'].map({
    'Within past year (anytime less than 12 months ago)': 0,
    'Within past 2 years (1 year but less than 2 years ago)': 1,
    'Within past 5 years (2 years but less than 5 years ago)': 2,
    '5 or more years ago': 3,
})

df_2022['CovidPos'] = df_2022['CovidPos'].map({
    'No': 0,
    'Yes': 1,
})

In [122]:
SEED = 67

X = df_2022.drop(['HeartDisease', 'HadAngina', 'HadHeartAttack'], axis=1)
y = df_2022['HeartDisease']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    train_size=0.8,
    random_state=SEED,
    stratify=y
)

X_valid, X_test, y_valid, y_test = train_test_split(
    X_test,
    y_test,
    train_size=0.5,
    random_state=SEED,
    stratify=y_test
)

In [123]:
transformer2022 = ColumnTransformer(
    transformers=[
        ('categories', OneHotEncoder(handle_unknown='ignore'), ['State',
                                                                'Sex',
                                                                'HadDiabetes',
                                                                'ECigaretteUsage',
                                                                'RaceEthnicityCategory',
                                                                'TetanusLast10Tdap',]),
        ('numericals', Pipeline([
            ('nans', SimpleImputer(strategy='median', add_indicator=True)),
        ]), list(set(X_test.columns) - set(['State',
                                            'Sex',
                                            'HadDiabetes',
                                            'ECigaretteUsage',
                                            'RaceEthnicityCategory',
                                            'TetanusLast10Tdap',
                                            'HadAngina',
                                            'HadHeartAttack'])))
    ]
)

model2022 = Pipeline([
    ('transformer', transformer2022),
    ('model', RandomForestClassifier(
        class_weight='balanced',
        random_state=SEED,
        n_jobs=-1,
        max_depth=15,
        min_samples_split=20,
        min_samples_leaf=10,
    ))
])

model2022.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('transformer', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('categories', ...), ('numericals', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different tr

In [124]:
y_proba2022 = model2022.predict_proba(X_test)[:, 1]
y_pred2022 = model2022.predict(X_test)

precision, recall, thresholds = precision_recall_curve(y_test, y_proba2022)

metrics_2022 = {
    'data': '2022',
    'model': 'RandomForestClassifier',
    'roc_auc_score': roc_auc_score(y_test, y_proba2022),
    'f1_score': f1_score(y_test, y_pred2022),
    'pr_auc_score': auc(recall, precision),
    'balanced_accuracy_score': balanced_accuracy_score(y_test, y_pred2022)
}

In [125]:
results = pd.DataFrame([
    metrics2020,
    metrics_2022,
    metrics_merged,
])

In [126]:
results.to_csv(r'../results/random_forest.csv')